In [ ]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd()))
import time
import json
import numpy as np
import matplotlib.pyplot as plt

from pc import PS
from modules import ADC,DAC,CHIP
from command import CMD,CmdData,Packet
from command.singleCmdInfo import *

from util import plot_v_cond,plot_cond,show_crossbar

In [ ]:
chip=CHIP(PS(host="192.168.1.10", port = 7, debug=0),init=True)
chip.set_device_cfg(deviceType=0)
chip.add_compiler("./code/")

In [ ]:
need_read = np.zeros((256,256))
need_read[180:200,100:120]=1
# for pulse_width in [1e-6,2e-6,5e-6,10e-6]:
for pulse_width in [1e-6]:
    # for v in [0.6]:
    for v in [0.6,0.7,0.8,0.9,1.0]:
        ans = []
        for i in range(1):
            # 先reset再读
            chip.write_point2(crossbar=need_read,write_voltage=1.1,tg=5,pulse_width=1e-6,set_device=False)
            voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            cond_sub_base1 = chip.voltage_to_cond(voltage-voltage_base)
            plot_cond(cond_sub_base1,title="cond_sub_base1")
            cond_sub_base1 = cond_sub_base1.flatten()

            # 再set再读
            chip.write_point2(crossbar=need_read,write_voltage=v,tg=3,pulse_width=pulse_width,set_device=True)
            voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            cond_sub_base2 = chip.voltage_to_cond(voltage-voltage_base)
            plot_cond(cond_sub_base2,title="cond_sub_base2")
            cond_sub_base2=cond_sub_base2.flatten()

            # 这里有除0错误，加了一个非常小的值
            ans3 = ((cond_sub_base2-cond_sub_base1)/(cond_sub_base1)).flatten()

            tmp=np.vstack((cond_sub_base1,cond_sub_base2,ans3))

            ans.append(tmp)
        np.save(f"./result/ncep1/pulse_width={(int(pulse_width*1e6))}_v={int(v*10)}",ans)

In [ ]:
need_read = np.zeros((256,256))
need_read[180:200,100:120]=1
for pulse_width in [1e-6,2e-6,5e-6,10e-6]:
# for pulse_width in [1e-6]:
    # for v in [2]:
    for v in [1.0,1.1,1.2,1.3,1.4]:
        ans = []
        for i in range(10):
            # set再读
            chip.write_point2(crossbar=need_read,write_voltage=2,tg=3,pulse_width=1e-6,set_device=True)
            voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            cond_sub_base2 = chip.voltage_to_cond(voltage-voltage_base)
            plot_cond(cond_sub_base2,title="cond_sub_base2")
            cond_sub_base2=cond_sub_base2.flatten()

            # reset再读
            chip.write_point2(crossbar=need_read,write_voltage=v,tg=5,pulse_width=pulse_width,set_device=False)
            voltage_base = chip.read_point2(crossbar=need_read, read_voltage=0,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            voltage = chip.read_point2(crossbar=need_read, read_voltage=0.1,tg=5,gain=1,from_row=True,out_type=0)[180:200,100:120]
            cond_sub_base1 = chip.voltage_to_cond(voltage-voltage_base)
            plot_cond(cond_sub_base1,title="cond_sub_base1")
            cond_sub_base1 = cond_sub_base1.flatten()

            # 这里有除0错误，加了一个非常小的值
            ans3 = ((cond_sub_base2-cond_sub_base1)/(cond_sub_base1)).flatten()

            tmp=np.vstack((cond_sub_base1,cond_sub_base2,ans3))

            ans.append(tmp)
        np.save(f"./result/ncep1/pulse_width={(int(pulse_width*1e6))}_v={int(v*10)}",ans)

In [ ]:
data = np.load("./result/ncep1/pulse_width=1_v=8.npy")
# 处理除0错误的值
data[data>1e10] = 0
data[data<-1e10] = 0
print(data.shape)
plt.plot(data[0,0,:])
plt.show()
plt.plot(data[0,1,:])
plt.show()
plt.plot(data[0,2,:])
plt.ylim(-1, 1)
plt.show()

# 提取 y 轴数据（假设是 data[0, 2, :]）
y_data = data[0, 2, :]
# 统计 y 在 [-1, 1] 区间内的数据点数量
count = np.sum((y_data > 0) & (y_data <= 1))
print(f"y 在 [0, 1] 区间内的数据点数量为: {count}")

In [ ]:
# 文件夹路径
folder_path = "./result/ncep1/"
# 获取文件夹中所有的 .npy 文件
files = [f for f in os.listdir(folder_path) if f.endswith('.npy') and 'pulse_width=2_' in f]
# 初始化一个列表来存储 x 轴标签和对应的 y 轴数据
x_labels = []
y_data = []
for k in range(0, 10):
    # 遍历所有文件
    for file in files:
        # 读取文件
        data = np.load(os.path.join(folder_path, file))
        
        # 处理除0错误的值
        data[data > 1e10] = 0
        data[data < -1e10] = 0
        
        # 提取文件名中的 v 值作为 x 轴标签
        v_value = float(file.split('_v=')[1].split('.')[0])
        x_labels.append(v_value)
        
        # 提取 y 轴数据（假设是 data[0, 2, :])
        y_data.append(data[k, 2, :])

    # 绘制所有曲线
    # for i, y in enumerate(y_data):
    #     plt.plot([x_labels[i]] * len(y), y)

    # 为每个数据集创建一个箱式图
    for i, y in enumerate(y_data):
        plt.scatter([x_labels[i]]*len(y), y, color='black', s=10)  # 添加散点图

    # 设置 y 轴范围
    plt.ylim(0,5)
    # 显示图形
    plt.show()